In [1]:
import sys
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [2]:
# 1) Add the *project root* (the parent of "src") to sys.path
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
sys.path.append(src_path)

# 2) Import from the package path
from a_func_generate_dfs_for_db import * 

In [3]:
pd.set_option("display.max_colwidth", 100)  # Begrenze Spaltenbreite auf 100 Zeichen


# Hauptverzeichnisse
data_nzz_path = "../data/NZZ_original"              # JSON-Ordner
data_nzz_csv_path = "../data/data_nzz_csv"      # Ziel-CSV-Dateien
meta_data_path = "../data/nzz_metadata.csv"        # Übersichtstabelle als CSV-Datei

# Ausgabeordner erzeugen, falls nicht vorhanden
os.makedirs(data_nzz_csv_path, exist_ok=True)

Here’s the translation and improved version:

For each plot, a CSV file is generated, structured like an Excel file, in order to create a plot. The plot's ID is used as the name for the CSV file, allowing easy access to the file later. It should still be verified whether the content of the files makes sense.

The output file is a CSV file, with one row per plot. The following features from the JSON file are stored:
- id
- title
- subtitle
- created_date
- interval
- chart_type

It should be checked whether additional columns are required.

To open jason file for a specific ID enter: notepad data/<ID>/<ID>.json
Example: notepad data/0021acd20bdafd2347513985db15a2d9/0021acd20bdafd2347513985db15a2d9.json


## nzz_metadata und nzz_csv erstellen

In [4]:
generate_metadata_and_csv_per_plot(
    data_nzz_path=data_nzz_path, 
    data_nzz_csv_path=data_nzz_csv_path,
    meta_data_path=meta_data_path)

✅ Einzel-CSV-Dateien gespeichert in: '../data/data_nzz_csv'
✅ Übersichtstabelle gespeichert als: '../data/nzz_metadata.csv'


### nzz_metadata.csv untersuchen

In [5]:
nzz_metadata = pd.read_csv(meta_data_path)
nzz_metadata.head()

,id,title,subtitle,notes,chart_type,x_category,num_columns,num_rows,x_axis_type,y_axis_type,x_axis_min,x_axis_max,y_axis_min,y_axis_max,highlighted_col,highlighted_row,prognosis,events,complex
0,0023e8fed9d111fed753bb3f6b0afe78,Auch der Handel mit der EU mit Autoteilen ist aus Sicht der USA defizitär,in Mrd. $,NaN,Bar,Jahr,3,6,numeric,numeric,2012,2017,5.011559e+00,2.079093e+01,[],[],NaN,NaN,True
1,0057a266a9c555637dabccece11c5468,Besucher am Züri-Fäscht,NaN,NaN,Bar,X,2,8,numeric,numeric,1998,2019,1.500000e+06,2.500000e+06,[],[],NaN,NaN,False
2,005e2925d4b679defb5afa4daef05ee5,"59,1 % der Italiener sagen «Nein» zur Verfassungsreform",Ergebnis der Volksabstimmung in Prozent,NaN,Bar,X,2,2,categorical,numeric,NaN,NaN,4.090000e+01,5.910000e+01,[],[],NaN,NaN,False
3,0062e0b6b75d81b20a18a5ab938ada72,Abkehr von Treasuries,"Rendite 10jährige US-Staatsanleihen, in %",NaN,Line,X,2,241,categorical,numeric,02/01/2017,04/12/2017,2.040000e+00,2.630000e+00,[],[],NaN,NaN,False
4,006f6723bd41797d2bb1e3f65444757f,SMI avanciert kräftig,Kursentwicklung in Punkten,NaN,Line,X,2,251,categorical,numeric,01/03/2017,12/29/2017,8.229010e+03,9.452320e+03,[],[],NaN,NaN,False


In [6]:
# print all charts which are complex
complex_nzz_metadata = nzz_metadata[nzz_metadata['complex'] == True]  
simple_nzz_metadata = nzz_metadata[nzz_metadata['complex'] == False] 

main_folder = "../data/NZZ_original"

def main(ids: List[str] = None):

    for folder in os.listdir(main_folder):
        if folder in ids:
            folder_path = os.path.join(main_folder, folder)
            if not os.path.isdir(folder_path):
                continue

            json_file_path = os.path.join(folder_path, f"{folder}.json")
            if not os.path.isfile(json_file_path):
                continue
            
            image_path = os.path.join(folder_path, f"{folder}.png")
            if os.path.exists(image_path):
            # Bild laden
                img = mpimg.imread(image_path)
                
                # Bild anzeigen
                plt.imshow(img)
                plt.axis('off')  # Verhindert die Anzeige der Achsen
                plt.show()
            else:
                print(f"Datei unter {image_path} existiert nicht.")

Es gibt noch einen Spezialfall: 022a672b3f9fa2b1be4ea3c48ad72ca1
Dieser hat 2 Linien, welche aber den anschein machen als wären sie hintereinander gestapelt. Sollte simple sein, ist hier jedoch als complex markiert.

Jahr	Regierungschef	StaatsprÃ¤sident
2004	15161.62	
2005	16309.68	
2006	17250.77	
2007	17901.48	
2008	17836.81	
2009	16783.44	
2010	17959.26	
2011	19660.89	
2012	20282.03	
2013	21650.76	
2014	22401.88	22401.88
2015		23382.25
2016		23679.4
2017		
2018		

In [7]:
# main(ids=complex_nzz_metadata['id'].tolist())

In [8]:
cols_to_replace = ["highlighted_row", "highlighted_col", "events"]
for col in cols_to_replace:
    nzz_metadata[col] = nzz_metadata[col].replace("[]", np.nan)

## df_data_table erstellen 

(ohne "x_category", "events" und "prognosis")

Bisherigen Spalten:
['chart_id', 'x_value', 'y_value', 'y_category', 'row_index', 'col_index']

In [9]:
df_data_flat= format_csv_for_db(csv_path=data_nzz_csv_path)

df_data_flat.columns.tolist()

['chart_id', 'x_value', 'y_value', 'y_category', 'row_index', 'col_index']

## df_events_table erstellen + nzz_metadata erweitern

nzz_metadata erweitern mit "events" 

In [10]:
id = "014e6bf3927ddaf505c8053e37c8b688"

# Now print
print(nzz_metadata.loc[nzz_metadata["id"] == id].events)

19    [{'type': 'point', 'date': '2019-12-29', 'label': 'Beginn der Corona-Pandemie in China'}]
Name: events, dtype: object


In [11]:
df_event = nzz_metadata.loc[nzz_metadata.events.notna(), ["id", "events"]]

# Apply ast.literal_eval to convert string to Python list
nzz_metadata["events"] = nzz_metadata["events"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [12]:
expanded_rows = []

for _, row in nzz_metadata.iterrows():
    id = row["id"]
    events = row["events"]
    
    if isinstance(events, list):
        for event in events:
            event_type = event.get("type")
            if event_type == "point":
                event_date = event.get("date")
                event_label = event.get("label")
                expanded_rows.append({
                    "chart_id": id,
                    "type": event_type,
                    "date": event_date,
                    "label": event_label
                })
            elif event_type == "range":
                event_date_from = event.get("dateFrom")
                event_date_to = event.get("dateTo")
                event_label = event.get("label")
                expanded_rows.append({
                    "chart_id": id,
                    "type": event_type,
                    "date": f"{event_date_from}–{event_date_to}",
                    "label": event_label
                })

# Neues DataFrame erstellen
df_event_table = pd.DataFrame(expanded_rows)

# Ergebnis
print(df_event_table.head(2).T)

                                            0  \
chart_id     014e6bf3927ddaf505c8053e37c8b688   
type                                    point   
date                               2019-12-29   
label     Beginn der Corona-Pandemie in China   

                                                  1  
chart_id           0154a0854ad542755c62796906aadb08  
type                                          point  
date                                      20.1.2017  
label     20. 1. 2017: Donald Trump wird Präsident.  


## df_chart_table erstellen + df_data_table ergänzen

df_data_table mit "x_category", "prognosis" und "highlighted" ergänzen.

In [13]:
df_all = df_data_flat.merge(nzz_metadata, left_on="chart_id", right_on="id", how="left")

In [14]:
df_all = get_highlighted_col(df_all)

df_all = mark_prognosis(df_all)

df_all = df_all.drop(["num_columns", "num_rows", "highlighted_col", "highlighted_row", "events"], axis=1)

In [15]:
# df_chart_table kreieren
main_table_col = ["id", "chart_type", "title", "subtitle", "notes", "complex", "x_axis_type", "y_axis_type", "x_axis_min", "x_axis_max", "y_axis_min", "y_axis_max"]
df_chart_table = df_all.groupby("chart_id", as_index=False).first()
df_chart_table = df_chart_table[main_table_col]

# df_data_table_NZZ mit "x_category", "prognosis" und "highlighted" ergänzen
data_table_col = ["chart_id", "x_category", "x_value", "y_category", "y_value", "row_index", "col_index", "prognosis", "highlighted"]
df_data_table_NZZ = df_all[data_table_col].reset_index().rename(columns={"index": "id"})

## df_chart_type_table erstellen und df_chart_table anpassen

In [16]:
unique_charts = df_chart_table["chart_type"].dropna().unique()

df_chart_type_table = pd.DataFrame({
    "id": range(len(unique_charts)),
    "type": unique_charts
})

# Mapping erstellen
chart_type_map = dict(zip(df_chart_type_table["type"], df_chart_type_table["id"]))

# IDs zuweisen
df_chart_table["chart_type_id"] = df_chart_table["chart_type"].replace(chart_type_map)

C:\Users\Julia\AppData\Local\Temp\ipykernel_10868\717908755.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_chart_table["chart_type_id"] = df_chart_table["chart_type"].replace(chart_type_map)


## Alle genrierten Tables abspeichern

In [17]:
tables = [df_chart_table, df_data_table_NZZ, df_event_table, df_chart_type_table]

In [18]:
for i, table in enumerate(tables):
    if isinstance(table, pd.DataFrame):
        print(f"\n--- Tabelle {i} ---")
        print(table.head(1).T)
    else:
        print(f"\nWarnung: Element {i} ist keine DataFrame: {type(table)}")


--- Tabelle 0 ---
                                                                                       0
id                                                      0023e8fed9d111fed753bb3f6b0afe78
chart_type                                                                           Bar
title          Auch der Handel mit der EU mit Autoteilen ist aus Sicht der USA defizitär
subtitle                                                                       in Mrd. $
notes                                                                               None
complex                                                                             True
x_axis_type                                                                      numeric
y_axis_type                                                                      numeric
x_axis_min                                                                          2012
x_axis_max                                                                          2017
y_

In [19]:
# Ausgabeordner erzeugen, falls nicht vorhanden
os.makedirs("../data/db_tables", exist_ok=True)


df_chart_table.to_csv("../data/db_tables/df_chart_table.csv")


df_data_table_NZZ.to_csv("../data/db_tables/df_data_table.csv")


df_event_table.to_csv("../data/db_tables/df_event_table.csv")


df_chart_type_table.to_csv("../data/db_tables/df_chart_type_table.csv")

## df_modell_table erstellen

In [20]:
cols = ["company", "model_series", "created", "URL"]

df_model_table = pd.DataFrame({
    "company": [ 
        "Meta",
        "google",
        "OpenAI",
        "SentenceTransformers"
        ],
    "model_series": [
        "meta-llama/llama-4-maverick:free",
        "google/gemini-2.5-flash-preview-09-2025",
        "openai/o4-mini",
        "sentence-transformers/all-MiniLM-L6-v2"
        ],
    "created": [
        "05.04.2025",
        "25.09.2025",
        "16.04.2025",
        "30.08.2021"
        ],
    "URL": [
        "https://openrouter.ai/meta-llama/llama-4-maverick:free",
        "https://statics.teams.cdn.office.net/evergreen-assets/safelinks/2/atp-safelinks.html",
        "https://openrouter.ai/openai/o4-mini",
        "https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2"
        ],
})

df_model_table.to_csv("../data/db_tables/df_model_table.csv")

## df_golden_summary_table erstellen

Hierfür müssen wir die Texte haben. Sollten wir wohl in einem JSON File speichern.